# Red Five — conditional uncertainty, one component at a time

Synthetic autocorrelated scores and targets. Explicit moving-block intervals for per-contract temporal correlation. They are conditional on stationarity/dependence and time-grid assumptions, not selection-adjusted and not acceptance verdicts. The synthetic date window is **not** a locked final assessment.

In [ ]:
%matplotlib inline
from pathlib import Path
from uuid import uuid4

from IPython.display import display
from matplotlib.figure import Figure

from red_five.component_export import Component, export_components
from red_five.composition import PlotOptions, Selection
from red_five.rendering import verify_bundle
from red_five.temporal import FoldSpec
from red_five.trials import TrialLedger
from red_five.uncertainty import BootstrapConfig

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "examples/uncertainty-signals.csv").is_file()
)
signals = (ROOT / "examples/uncertainty-signals.csv").read_bytes()
config = (ROOT / "examples/uncertainty-evaluation.json").read_bytes()
plan = (ROOT / "STATISTICAL_ANALYSIS_PLAN.md").read_bytes()
lock = (ROOT / "uv.lock").read_bytes()
fold = FoldSpec(
    "synthetic-evaluation",
    "2025-01-01T00:00:00Z",
    "2025-03-01T00:00:00Z",
    "2025-07-20T00:00:00Z",
)
ledger = TrialLedger(ROOT / "build" / "uncertainty-trials.sqlite")
attempt = uuid4().hex[:12]
policy = BootstrapConfig(
    metric="pearson_ic",
    block_length=8,
    replicates=500,
    seed=372,
    confidence=0.95,
    step_seconds=86400,
    minimum_time_points=64,
)
result = ledger.run(
    f"{attempt}-pearson",
    "synthetic-uncertainty-family",
    "uncertainty",
    signals,
    config,
    plan,
    lock,
    fold=fold,
    uncertainty=policy,
)

## Exact table, then one model's chart

Dots are point estimates; vertical segments are conditional percentile intervals. Unavailable intervals are annotated instead of replaced by zeros. The `estimate`, `lower` and `upper` columns travel together in chart exports. Inspect all count/reason columns before interpreting an interval.

In [ ]:
display(result.select().table())
es = result.select(Selection(model_ids=("ES-model",), precision=4))
display(es.figure(options=PlotOptions(title="ES temporal Pearson correlation")))

## Metadata and your subplot layout

The policy records every resampling setting. Paired scores/returns are resampled together within a contract; unrelated contracts never share a fit. Missing/immature time points or gaps in the declared elapsed-time grid make the interval unavailable. This regular-grid first version does not silently reinterpret exchange holidays.

In [ ]:
display(result.diagnostics["policy"])
figure = Figure(figsize=(11, 12), layout="constrained")
axes = figure.subplots(2, 1)
for ax, model in zip(axes, ("ES-model", "NQ-model"), strict=True):
    result.select(Selection(model_ids=(model,))).plot(
        ax=ax, options=PlotOptions(title=f"{model}: conditional interval")
    )
display(figure)

## Changing the estimator is a new trial, not styling

Spearman ranks are recomputed inside each resample. Comparing variants after viewing outcomes is exploration; neither interval is adjusted for that selection.

In [ ]:
rank = ledger.run(
    f"{attempt}-rank",
    "synthetic-uncertainty-family",
    "uncertainty",
    signals,
    config,
    plan,
    lock,
    fold=fold,
    uncertainty=BootstrapConfig(
        metric="rank_ic",
        block_length=8,
        replicates=500,
        seed=372,
        confidence=0.95,
        step_seconds=86400,
        minimum_time_points=64,
    ),
)
display(rank.select().table())
display(
    rank.select().figure(options=PlotOptions(title="Temporal Spearman correlations"))
)
display(ledger.table().dataframe())

## Export only chosen components

Style changes reuse evidence. Changing a block length, seed, confidence, metric or sample requires a new computation and trial. Direct custom Matplotlib edits are exploratory and are not replayed by standard component exports.

In [ ]:
output = ROOT / "build" / f"uncertainty-components-{attempt}"
export_components(
    [
        Component(es, "uncertainty", PlotOptions(title="ES Pearson interval")),
        Component(rank.select(), "uncertainty", PlotOptions(title="Rank intervals")),
    ],
    output,
)
assert verify_bundle(output)["scope"] == "partial"
print(output)

## Before consequential assessment

Read `docs/FINAL_ASSESSMENT.md`: actual holdout dates, hypothesis family/history, access authority and rejection/acceptance rules still need owner decisions. No enforced holdout lock exists. Cross-sectional mode (same API) resamples the time series of per-date ICs and estimates their equal-date mean, not a pooled security-level correlation. See `docs/UNCERTAINTY.md` for method references, assumptions, limits and the deliberately limited simulation evidence.